# Task 1

In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
import heapq
from abc import ABC, abstractmethod

## Task 1.1 – Discretización del Mundo

In [2]:
# La imagen tiene miles de píxeles. En lugar de tratar cada píxel como un nodo (ineficiente), la dividimos en "tiles" de `tile_size x tile_size` píxeles.
# Cada tile se convierte en 
# un nodo del grafo.

FREE  = 0   # blanco (libre)
WALL  = 1   # negro (pared)
START = 2   # rojo (inicio)
GOAL  = 3   # verde (metas)

def classify_tile(tile_pixels: np.ndarray) -> int:
    # Promedio de R, G, B en todos los píxeles del tile
    avg = tile_pixels.mean(axis=(0, 1))  # shape: (3,)
    r, g, b = avg[0], avg[1], avg[2]
    
    # Clasificacion de color
    if r < 50 and g < 50 and b < 50:
        return WALL
    elif r > 150 and g < 80 and b < 80:
        return START
    elif g > 150 and r < 80 and b < 80:
        return GOAL
    else:
        return FREE


def discretize_image(image_path: str, tile_size: int = 10):
    # 1. Cargar imagen y convertir a RGB
    img = Image.open(image_path).convert('RGB')
    img_array = np.array(img)  # shape: (alto, ancho, 3)
    H, W, _ = img_array.shape
    print(f"Imagen cargada: {W}x{H} píxeles")
    
    # 2. Calcular dimensiones de la grilla
    rows = H // tile_size
    cols = W // tile_size
    print(f"Grilla discreta: {rows}x{cols} nodos (tile_size={tile_size})")
    
    # 3. Construir la matriz
    grid = np.zeros((rows, cols), dtype=int)
    start = None
    goals = []
    
    for r in range(rows):
        for c in range(cols):
            # Extraer los píxeles de este tile
            r0, r1 = r * tile_size, (r + 1) * tile_size
            c0, c1 = c * tile_size, (c + 1) * tile_size
            tile = img_array[r0:r1, c0:c1]  # shape: (tile_size, tile_size, 3)
            
            # Clasificar el tile
            cell_type = classify_tile(tile)
            grid[r, c] = cell_type
            
            # Registrar inicio y metas
            if cell_type == START:
                start = (r, c)
            elif cell_type == GOAL:
                goals.append((r, c))
    
    print(f"Inicio encontrado en: {start}")
    print(f"Metas encontradas en: {goals}")
    
    return grid, start, goals, img_array


def visualize_grid(grid: np.ndarray, title: str = "Grilla Discreta"):
    # Mapa de colores: 0=blanco, 1=negro, 2=rojo, 3=verde
    color_map = {
        FREE:  [1.0, 1.0, 1.0],   # Blanco
        WALL:  [0.0, 0.0, 0.0],   # Negro
        START: [1.0, 0.0, 0.0],   # Rojo
        GOAL:  [0.0, 0.8, 0.0],   # Verde
    }
    
    rows, cols = grid.shape
    img_vis = np.ones((rows, cols, 3))
    for cell_type, color in color_map.items():
        mask = grid == cell_type
        img_vis[mask] = color
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_vis, interpolation='nearest')
    plt.title(title)
    plt.axis('off')
    
    # Leyenda
    patches = [
        mpatches.Patch(color='white', label='Libre', edgecolor='gray'),
        mpatches.Patch(color='black', label='Pared'),
        mpatches.Patch(color='red',   label='Inicio'),
        mpatches.Patch(color='green', label='Meta'),
    ]
    plt.legend(handles=patches, loc='upper right')
    plt.tight_layout()
    plt.show()